# Entra Agent ID setup (Lab1)

Step-by-step notebook: **Agent ID** (blueprint + principal + secret + agent) and **Weather API resource** (app + URI + service principal).

## Prerequisites

- Entra tenant with Agent ID enabled; permission to create agent blueprints (role or tenant setting).
- Python 3.10+ and Jupyter (VS Code / Cursor).
- **Admin sign-in** uses **device code** in the notebook (same public client as `Connect-MgGraph`). 
- Graph **delegated** permissions (admin consent if required): Agent Identity blueprint scopes, `User.Read`, and **`Application.ReadWrite.All`** for the Weather API app (requires Application Administrator or equivalent). See [Create blueprint](https://learn.microsoft.com/en-us/entra/agent-id/create-blueprint).
- After changing scopes in the config cell, **re-run admin sign-in (device code)** so your token includes `Application.ReadWrite.All`.
- Install dependencies: `pip install -r requirements.txt` (in this folder).

## Security

**Do not commit this notebook with outputs saved** — Step 2 shows a client secret once.

## Graph API version

This lab uses **Microsoft Graph v1.0** endpoints. If a call fails with v1.0, see [Create blueprint](https://learn.microsoft.com/en-us/entra/agent-id/create-blueprint) and try the equivalent beta URL only as a fallback.

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import random
import time
from datetime import datetime, timedelta, timezone

import httpx
import msal

TENANT_ID = "293882e2-ff46-4747-b46a-a84ca8e98fa0"  # @@@ YOUR TENANT ID @@@
RUN_SUFFIX = random.randint(100000, 999999)
BLUEPRINT_NAME = f"Lab1 Agent Blueprint {RUN_SUFFIX}"
AGENT_NAME = f"Lab1 Weather Agent {RUN_SUFFIX}"
print(f"Run suffix: {RUN_SUFFIX}")
print(f"Blueprint name: {BLUEPRINT_NAME}")
print(f"Agent name: {AGENT_NAME}")

WEATHER_API_APP_ID = None  # optional: reuse an existing Weather API application (client id)
WEATHER_APP_DISPLAY_NAME = f"Entra Agent ID Sample - Weather API {RUN_SUFFIX}"
print(f"Weather app name: {WEATHER_APP_DISPLAY_NAME}")

GRAPH_ROOT = "https://graph.microsoft.com/v1.0"
# Microsoft Graph PowerShell app — same client ID as Connect-MgGraph.
GRAPH_POWERSHELL_APP_CLIENT_ID = "14d82eec-204b-4c2f-b7e8-296a70dab67e"

DELEGATED_SCOPES = [
    "https://graph.microsoft.com/AgentIdentityBlueprint.Create",
    "https://graph.microsoft.com/AgentIdentityBlueprint.AddRemoveCreds.All",
    "https://graph.microsoft.com/AgentIdentityBlueprintPrincipal.Create",
    "https://graph.microsoft.com/User.Read",
    "https://graph.microsoft.com/Application.ReadWrite.All",
]

if "00000000" in TENANT_ID:
    raise ValueError("Set TENANT_ID in the configuration cell before continuing.")

In [ ]:
def acquire_delegated_token_device_code(
    tenant_id: str, scopes: list[str], client_id: str
) -> str:
    """Device code flow for Entra Agent Identity Graph scopes (steps 1–2)."""
    authority = f"https://login.microsoftonline.com/{tenant_id}"
    app = msal.PublicClientApplication(client_id, authority=authority)
    flow = app.initiate_device_flow(scopes=scopes)
    if "user_code" not in flow:
        raise RuntimeError(f"Device flow failed: {flow}")
    print(flow["message"])
    result = app.acquire_token_by_device_flow(flow)
    if "access_token" not in result:
        raise RuntimeError(f"Sign-in failed: {result.get('error_description', result)}")
    return result["access_token"]


def graph_request(
    method: str,
    url: str,
    token: str,
    json_body: dict | None = None,
    extra_headers: dict | None = None,
) -> httpx.Response:
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
        "OData-Version": "4.0",
    }
    if extra_headers:
        headers.update(extra_headers)
    with httpx.Client(timeout=60.0) as client:
        response = client.request(method, url, headers=headers, json=json_body)
    if response.status_code >= 400:
        raise RuntimeError(f"{method} {url} -> {response.status_code}: {response.text}")
    return response


def acquire_blueprint_token(
    tenant_id: str,
    app_id: str,
    client_secret: str,
    retries: int = 5,
    delay_seconds: float = 3.0,
) -> str:
    """Client credentials for blueprint (step 3)."""
    authority = f"https://login.microsoftonline.com/{tenant_id}"
    app = msal.ConfidentialClientApplication(
        app_id,
        authority=authority,
        client_credential=client_secret,
    )
    last_error = None
    for attempt in range(1, retries + 1):
        result = app.acquire_token_for_client(scopes=["https://graph.microsoft.com/.default"])
        if "access_token" in result:
            return result["access_token"]
        last_error = result.get("error_description") or result
        if attempt < retries:
            print(f"Token not ready (attempt {attempt}/{retries}), waiting {delay_seconds}s...")
            time.sleep(delay_seconds)
    raise RuntimeError(f"Failed to acquire blueprint token: {last_error}")

## Admin sign-in (device code)

Run the cell below and complete the sign-in in your browser (URL + user code). Uses `GRAPH_POWERSHELL_APP_CLIENT_ID` — same permissions model as `Connect-MgGraph`. If your tenant blocks this client, register your own app and replace that value in the config cell.

In [ ]:
admin_token = acquire_delegated_token_device_code(
    TENANT_ID, DELEGATED_SCOPES, GRAPH_POWERSHELL_APP_CLIENT_ID
)
me = graph_request("GET", f"{GRAPH_ROOT}/me", admin_token).json()
user_id = me["id"]
print(f"Signed in as: {me.get('userPrincipalName', me.get('displayName'))}")
print(f"User object id (sponsor): {user_id}")

## Step 1 — Blueprint + blueprint principal

Creates the agent identity blueprint and its blueprint principal in the tenant.

In [ ]:
blueprint_body = {
    "displayName": BLUEPRINT_NAME,
    "sponsors@odata.bind": [f"{GRAPH_ROOT}/users/{user_id}"],
    "owners@odata.bind": [f"{GRAPH_ROOT}/users/{user_id}"],
}

blueprint_url = f"{GRAPH_ROOT}/applications/microsoft.graph.agentIdentityBlueprint"
blueprint_resp = graph_request("POST", blueprint_url, admin_token, blueprint_body).json()

blueprint_app_id = blueprint_resp["appId"]
blueprint_object_id = blueprint_resp["id"]
print(f"Blueprint appId (client id): {blueprint_app_id}")
print(f"Blueprint object id: {blueprint_object_id}")

principal_body = {"appId": blueprint_app_id}
principal_url = f"{GRAPH_ROOT}/servicePrincipals/microsoft.graph.agentIdentityBlueprintPrincipal"
principal_resp = graph_request("POST", principal_url, admin_token, principal_body).json()
principal_id = principal_resp["id"]
print(f"Blueprint principal id: {principal_id}")

print("Waiting 5s for principal propagation...")
time.sleep(5)

## Step 2 — Client secret

Adds a password credential to the blueprint. **Copy `CLIENT_SECRET` now** — it is not shown again.

In [ ]:
secret_end = (datetime.now(timezone.utc) + timedelta(days=365)).strftime("%Y-%m-%dT%H:%M:%SZ")
password_body = {
    "passwordCredential": {
        "displayName": f"Agent ID Secret {BLUEPRINT_NAME}",
        "endDateTime": secret_end,
    }
}

add_password_url = f"{GRAPH_ROOT}/applications/{blueprint_object_id}/addPassword"
secret_resp = graph_request("POST", add_password_url, admin_token, password_body).json()
CLIENT_SECRET = secret_resp.get("secretText")
if not CLIENT_SECRET:
    raise RuntimeError(f"addPassword did not return secretText: {secret_resp}")

print("CLIENT_SECRET saved to variable (length):", len(CLIENT_SECRET))
print("\n--- CLIENT SECRET (copy now) ---\n")
print(CLIENT_SECRET)
print("\n--- end secret ---")

## Step 3 — Agent identity

Uses blueprint **client credentials** (not the admin token) to create the agent.

In [ ]:
blueprint_token = acquire_blueprint_token(TENANT_ID, blueprint_app_id, CLIENT_SECRET)

agent_body = {
    "displayName": AGENT_NAME,
    "agentIdentityBlueprintId": blueprint_app_id,
    "sponsors@odata.bind": [f"{GRAPH_ROOT}/users/{user_id}"],
}
agent_url = f"{GRAPH_ROOT}/servicePrincipals/microsoft.graph.agentIdentity"
agent_resp = graph_request("POST", agent_url, blueprint_token, agent_body).json()

agent_identity_app_id = agent_resp["appId"]
agent_identity_sp_id = agent_resp["id"]
print(f"Agent display name: {agent_resp.get('displayName', AGENT_NAME)}")
print(f"Agent appId: {agent_identity_app_id}")
print(f"Agent service principal id: {agent_identity_sp_id}")

## Part 2 — Weather API resource 

Registers the sample Weather API in Entra: **app**, **application ID URI** (`api://...`), and **service principal**. 

Requires `Application.ReadWrite.All` on your admin token.

### Weather Step 1 — App registration

In [ ]:
select_fields = "id,appId,displayName,identifierUris"

if WEATHER_API_APP_ID:
    lookup_url = (
        f"{GRAPH_ROOT}/applications?$filter=appId eq '{WEATHER_API_APP_ID}'"
        f"&$select={select_fields}"
    )
    lookup = graph_request("GET", lookup_url, admin_token).json()
    if not lookup.get("value"):
        raise RuntimeError(f"No application for WEATHER_API_APP_ID={WEATHER_API_APP_ID}")
    weather_app = lookup["value"][0]
    print(f"Using existing app: {weather_app['displayName']} ({weather_app['appId']})")
else:
    safe_name = WEATHER_APP_DISPLAY_NAME.replace("'", "''")
    lookup_url = (
        f"{GRAPH_ROOT}/applications?$filter=displayName eq '{safe_name}'"
        f"&$select={select_fields}"
    )
    lookup = graph_request("GET", lookup_url, admin_token).json()
    if lookup.get("value"):
        weather_app = lookup["value"][0]
        print(f"Found by display name: {weather_app['appId']}")
    else:
        create_body = {
            "displayName": WEATHER_APP_DISPLAY_NAME,
            "signInAudience": "AzureADMyOrg",
        }
        weather_app = graph_request(
            "POST", f"{GRAPH_ROOT}/applications", admin_token, create_body
        ).json()
        print(f"Created Weather API app: {weather_app['appId']}")
        time.sleep(5)

weather_app_object_id = weather_app["id"]
weather_api_app_id = weather_app["appId"]
weather_identifier_uris = list(weather_app.get("identifierUris") or [])
print(f"Weather app object id: {weather_app_object_id}")
print(f"Weather API appId (client id): {weather_api_app_id}")

### Weather Step 2 — Application ID URI

In [ ]:
weather_application_id_uri = f"api://{weather_api_app_id}"

if weather_application_id_uri not in weather_identifier_uris:
    new_uris = list(dict.fromkeys(weather_identifier_uris + [weather_application_id_uri]))
    graph_request(
        "PATCH",
        f"{GRAPH_ROOT}/applications/{weather_app_object_id}",
        admin_token,
        {"identifierUris": new_uris},
    )
    weather_identifier_uris = new_uris
    print(f"Set application ID URI: {weather_application_id_uri}")
else:
    print(f"Application ID URI already set: {weather_application_id_uri}")

### Weather Step 3 — Service principal

In [ ]:
sp_lookup_url = (
    f"{GRAPH_ROOT}/servicePrincipals?$filter=appId eq '{weather_api_app_id}'"
    "&$select=id,appId"
)
sp_lookup = graph_request("GET", sp_lookup_url, admin_token).json()

if sp_lookup.get("value"):
    weather_sp_id = sp_lookup["value"][0]["id"]
    print(f"Weather API service principal exists: {weather_sp_id}")
else:
    sp_created = graph_request(
        "POST",
        f"{GRAPH_ROOT}/servicePrincipals",
        admin_token,
        {"appId": weather_api_app_id},
    ).json()
    weather_sp_id = sp_created["id"]
    print(f"Created Weather API service principal: {weather_sp_id}")
    time.sleep(5)

## Summary

Use `lab_context` with other Lab1 scripts. **Do not commit outputs that contain `CLIENT_SECRET`.**

In [ ]:
lab_context = {
    "tenant_id": TENANT_ID,
    "run_suffix": RUN_SUFFIX,
    "blueprint_name": BLUEPRINT_NAME,
    "blueprint_app_id": blueprint_app_id,
    "blueprint_object_id": blueprint_object_id,
    "principal_id": principal_id,
    "user_id": user_id,
    "client_secret": CLIENT_SECRET,
    "agent_name": AGENT_NAME,
    "agent_identity_app_id": agent_identity_app_id,
    "agent_identity_sp_id": agent_identity_sp_id,
    "weather_api_app_id": weather_api_app_id,
    "weather_app_object_id": weather_app_object_id,
    "weather_sp_id": weather_sp_id,
    "weather_application_id_uri": weather_application_id_uri,
}

display_keys = {k: v for k, v in lab_context.items() if k != "client_secret"}
print("lab_context (secret omitted from print):")
for key, value in display_keys.items():
    print(f"  {key}: {value}")

print("\nAuth: Agent steps 1–2 = device code; Agent step 3 = MSAL blueprint client_credentials.")
print("Sidecar .env: AGENT_CLIENT_ID = agent_identity_app_id; WEATHER_API_APP_ID = weather_api_app_id")
print("Next: run the Update sidecar .env cells below, then docker compose from Lab1/sidecar.")

## Update sidecar `.env`

Writes Entra IDs and blueprint secret to [`../sidecar/.env`](../sidecar/.env) for `docker compose`. **Do not commit** `.env` with secrets.

In [ ]:
from pathlib import Path


def update_dotenv(path: Path, updates: dict[str, str]) -> None:
    lines: list[str] = []
    if path.is_file():
        lines = path.read_text(encoding="utf-8").splitlines()

    remaining = dict(updates)
    new_lines: list[str] = []
    for line in lines:
        matched = False
        for key, value in list(remaining.items()):
            if line.strip().startswith(f"{key}="):
                new_lines.append(f"{key}={value}")
                del remaining[key]
                matched = True
                break
        if not matched:
            new_lines.append(line)

    for key, value in remaining.items():
        if new_lines and new_lines[-1] != "":
            new_lines.append("")
        new_lines.append(f"{key}={value}")

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(new_lines) + "\n", encoding="utf-8")


def find_sidecar_env() -> Path:
    for sidecar_dir in (
        Path.cwd() / "sidecar",
        Path.cwd().parent / "sidecar",
        Path.cwd() / "Lab1" / "sidecar",
    ):
        if (sidecar_dir / "docker-compose.yml").is_file():
            return (sidecar_dir / ".env").resolve()
    raise FileNotFoundError("Could not find Lab1/sidecar (cwd should be Lab1/scripts or repo root)")


sidecar_env = find_sidecar_env()

env_updates = {
    "TENANT_ID": TENANT_ID,
    "BLUEPRINT_APP_ID": blueprint_app_id,
    "BLUEPRINT_CLIENT_SECRET": CLIENT_SECRET,
    "AGENT_CLIENT_ID": agent_identity_app_id,
    "WEATHER_API_APP_ID": weather_api_app_id,
}

update_dotenv(sidecar_env, env_updates)
print(f"Updated {sidecar_env}")
for key in env_updates:
    if key == "BLUEPRINT_CLIENT_SECRET":
        print(f"  {key}=***")
    else:
        print(f"  {key}={env_updates[key]}")